# Build the MBO Feature Table 

This notebook connects to the AWS EMR Spark cluster, loads the raw MBO CSV files from Amazon S3, and wrangles them into a clean 1-second feature table for modeling.

Using the Databento schema and flag definitions, the notebook:
- parses timestamps and decodes flag bits
- filters snapshot and administrative rows
- aggregates live events into 1-second buckets
- engineers side, size, and trade-based price features
- builds a dense 1-second session timeline
- computes log returns, realized volatility targets, and rolling 60-second features

The final feature table is saved back to S3 as a Parquet dataset for later EDA and modeling.


In [ ]:
# install packages
!pip uninstall -y dataproc-spark-connect pyspark
!pip install -q pyspark==3.5.6


Found existing installation: dataproc-spark-connect 1.0.2
Uninstalling dataproc-spark-connect-1.0.2:
  Successfully uninstalled dataproc-spark-connect-1.0.2
Found existing installation: pyspark 4.0.2
Uninstalling pyspark-4.0.2:
  Successfully uninstalled pyspark-4.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.4/317.4 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 12.4 MB/s eta 0:00:00


In [ ]:
# imports
import os
from getpass import getpass

In [ ]:
# set up AWS credentials - so EMR can access S3

os.environ["AWS_ACCESS_KEY_ID"] = getpass("AWS_ACCESS_KEY_ID: ").strip()
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("AWS_SECRET_ACCESS_KEY: ").strip()
###Needed for AWS Academy###
#os.environ["AWS_SESSION_TOKEN"] = getpass("AWS_SESSION_TOKEN: ").strip()
###Update to your bucket Region###
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"

AWS_ACCESS_KEY_ID: ··········
AWS_SECRET_ACCESS_KEY: ··········


In [ ]:
# get client ip for AWS setup
!curl -s https://checkip.amazonaws.com

35.201.249.211


In [ ]:
# Set to address of EMR stack
%set_env EMR_HOST=ec2-54-202-192-134.us-west-2.compute.amazonaws.com

env: EMR_HOST=ec2-54-202-192-134.us-west-2.compute.amazonaws.com


In [ ]:
from pyspark.sql import SparkSession
import os

# connect to the EMR Spark cluster
spark = SparkSession.builder \
  .appName("CIS-5450") \
  .remote("sc://{host}:15002".format(host=os.getenv('EMR_HOST'))) \
  .getOrCreate()


In [ ]:
# use UTC to match the MBO market-data timestamps, not local clock
spark.conf.set("spark.sql.session.timeZone", "UTC")
# set partitions
spark.conf.set("spark.sql.shuffle.partitions", "100")

In [ ]:
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import StringType, IntegerType, DoubleType, LongType


# define spark schema for MBO csv file using Databento schemas & data formats
# Note that we are loading timestamps as strings at first,
# since in metadata pretty_ts=true (ISO string) and timestamp includes nanoseconds
# Also, in metadata pretty_px=true so price is human-readable base 10
schema = StructType([
    StructField("ts_recv", StringType(), True),
    StructField("ts_event", StringType(), True),
    StructField("rtype", IntegerType(), True),
    StructField("publisher_id", IntegerType(), True),
    StructField("instrument_id", LongType(), True),
    StructField("action", StringType(), True),
    StructField("side", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("size", LongType(), True),
    StructField("channel_id", IntegerType(), True),
    StructField("order_id", LongType(), True),
    StructField("flags", IntegerType(), True),
    StructField("ts_in_delta", LongType(), True),
    StructField("sequence", LongType(), True),
    StructField("symbol", StringType(), True)
])

In [ ]:
# path for MBO csv files stored in S3
###CHANGE AS NEEDED###
BUCKET = "cis545-mbo-data-596916982631-us-west-2-an"
S3_PREFIX = "raw_MBO/"
S3_PATH = f"s3://{BUCKET}/{S3_PREFIX}*.mbo.csv"

In [ ]:
# read all MBO CSV files from S3 to spark df

sdf_raw = (
    spark.read
    .option("header", "true")
    .schema(schema)
    .csv(S3_PATH)
)

In [ ]:
# quick inspection
print("Columns:")
print(sdf_raw.columns)

print("First 5 rows:")
sdf_raw.show(5, truncate=False)



Columns:
['ts_recv', 'ts_event', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side', 'price', 'size', 'channel_id', 'order_id', 'flags', 'ts_in_delta', 'sequence', 'symbol']
First 5 rows:
+------------------------------+------------------------------+-----+------------+-------------+------+----+-----+----+----------+--------+-----+-----------+--------+------+
|ts_recv                       |ts_event                      |rtype|publisher_id|instrument_id|action|side|price|size|channel_id|order_id|flags|ts_in_delta|sequence|symbol|
+------------------------------+------------------------------+-----+------------+-------------+------+----+-----+----+----------+--------+-----+-----------+--------+------+
|2025-11-16T12:10:12.667167378Z|2025-11-16T12:10:12.489031171Z|160  |1           |42001597     |R     |N   |NULL |0   |19        |0       |8    |0          |0       |ZB.c.0|
|2025-11-16T12:10:12.692576485Z|2025-11-16T12:10:12.489031171Z|160  |1           |42296720     |R     |N   |

In [ ]:
# create view for spark SQL queries and make sdf_raw persistant
sdf_raw.createOrReplaceTempView("mbo_raw")

In [ ]:

# get total row count before any processing
query = """
SELECT
    COUNT(*) AS total_rows
FROM mbo_raw
"""

spark.sql(query).show()

+----------+
|total_rows|
+----------+
| 382479376|
+----------+


Raw CSV file: glbx-mdp3-20251201-20251215.mbo.csv - 43,405,386 rows, ~5.5 GB.

Raw CSV file: glbx-mdp3-20251216-20251231.mbo.csv - 47,863,703 rows, ~6.2 GB.

Raw CSV file: glbx-mdp3-20251115-20251130.mbo.csv - 135,013,176 rows, ~17.5 GB.

Raw CSV file: glbx-mdp3-20260101-20260115.mbo.csv - 156,197,111 rows, ~20.3 GB.

total rows for all MBO CSV files: 382,479,376

## Inspect raw MBO values

Drop null rows and inspect the key categorical columns
`action`, `side`, `symbol`, `flags`, `rtype`

According to Databento MBO schema:
- `rtype` should be 160 for MBO records
- `action` can represent Add, Cancel, Modify, cleaR book, Trade, Fill, or None
- `side` indicates the side that initiates the event
   and can be `B` (bid/buy), `A` (ask/sell), or `N` (no side specified)
- `flags` is a bit field that carries message and data-quality information,
  including snapshot-related indicators


In [ ]:
# check number of rows for each action code
# action can be Add, Cancel, Modify, cleaR book, Trade, Fill, or None.
query = """
SELECT
    action,
    COUNT(*) AS action_count
FROM mbo_raw
GROUP BY
    action
ORDER BY
    action_count DESC
"""

spark.sql(query).show()

+------+------------+
|action|action_count|
+------+------------+
|     A|   143250166|
|     C|   140791991|
|     M|    63103284|
|     F|    26429373|
|     T|     8904064|
|     R|         498|
+------+------------+


In [ ]:
# count rows for each side value
# side can be Ask/sell, Bid/buy, or None when no side is specified
query = """
SELECT
    side,
    COUNT(*) AS side_count
FROM mbo_raw
GROUP BY
    side
ORDER BY
    side_count DESC
"""

spark.sql(query).show()

+----+----------+
|side|side_count|
+----+----------+
|   B| 191057710|
|   A| 190428255|
|   N|    993411|
+----+----------+


In [ ]:
# count rows for each symbol
# ZT.c.0 = 2-Year Treasury
# ZF.c.0 = 5-Year Treasury
# ZN.c.0 = 10-Year Treasury
# ZB.c.0 = 30-Year Treasury Bond
# TN.c.0 = Ultra 10-Year Treasury
# UB.c.0 = Ultra Bond
# CL.c.0 = Crude Oil (WTI)
# BZ.c.0 = Brent Crude
# 6E.c.0 = Euro/USD FX

query = """
SELECT
    symbol,
    COUNT(*) AS symbol_count
FROM mbo_raw
GROUP BY
    symbol
ORDER BY
    symbol_count DESC
"""

spark.sql(query).show()

+------+------------+
|symbol|symbol_count|
+------+------------+
|ZN.c.0|    69836778|
|CL.c.0|    63514580|
|ZT.c.0|    51448866|
|ZF.c.0|    46223683|
|TN.c.0|    40593591|
|ZB.c.0|    35461136|
|UB.c.0|    28506768|
|6E.c.0|    28361797|
|BZ.c.0|    18532177|
+------+------------+


In [ ]:
# count rows for each flag value
# flags is a bit field that may indicate snapshot, last record, bad ts_recv, etc.

query = """
SELECT
    flags,
    COUNT(*) AS count
FROM mbo_raw
GROUP BY
    flags
ORDER BY
    count DESC
"""

spark.sql(query).show()

+-----+---------+
|flags|    count|
+-----+---------+
|  128|323570416|
|    0| 55253224|
|   40|  2505056|
|  132|  1020379|
|    4|   129743|
|  168|      473|
|    8|       85|
+-----+---------+


In [ ]:
# count rows for rtype value in MBO data
# rtype should be 160

query = """
SELECT
    rtype,
    COUNT(*) AS count
FROM mbo_raw
GROUP
    BY rtype
"""

spark.sql(query).show()

+-----+---------+
|rtype|    count|
+-----+---------+
|  160|382479376|
+-----+---------+


In [ ]:
# clean null rows in key columns
query = """
SELECT
    *
FROM mbo_raw
WHERE
    rtype = 160
  AND ts_event IS NOT NULL
  AND flags IS NOT NULL
  AND action IS NOT NULL
  AND symbol IS NOT NULL
"""

mbo_clean = spark.sql(query)
mbo_clean.createOrReplaceTempView("mbo_clean")

## Parse timestamps

The `ts_event` and `ts_recv` columns are loaded as strings because
`pretty_ts=true` in the metadata. This means we have ISO 8601
 timestamp strings with nanosecond precision.

Convert timestamp string columns into Spark timestamps for use in time-based processing.
Spark stores timestamps with microsecond precision.
Not an issue since aggregating events into 1-second buckets.

In [ ]:
# convert timestamp strings into Spark timestamps

query = """
SELECT
    *,
    TO_TIMESTAMP(ts_event) AS ts_event_parsed,
    TO_TIMESTAMP(ts_recv) AS ts_recv_parsed
FROM mbo_clean
"""

mbo_time_sdf = spark.sql(query)
mbo_time_sdf.createOrReplaceTempView("mbo_time")

mbo_time_sdf.show(5, truncate=False)

+------------------------------+------------------------------+-----+------------+-------------+------+----+-----+----+----------+--------+-----+-----------+--------+------+--------------------------+--------------------------+
|ts_recv                       |ts_event                      |rtype|publisher_id|instrument_id|action|side|price|size|channel_id|order_id|flags|ts_in_delta|sequence|symbol|ts_event_parsed           |ts_recv_parsed            |
+------------------------------+------------------------------+-----+------------+-------------+------+----+-----+----+----------+--------+-----+-----------+--------+------+--------------------------+--------------------------+
|2025-11-16T12:10:12.667167378Z|2025-11-16T12:10:12.489031171Z|160  |1           |42001597     |R     |N   |NULL |0   |19        |0       |8    |0          |0       |ZB.c.0|2025-11-16 12:10:12.489031|2025-11-16 12:10:12.667167|
|2025-11-16T12:10:12.692576485Z|2025-11-16T12:10:12.489031171Z|160  |1           |422967

In [ ]:
# confirm parsing of timestamps worked
query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE
        WHEN ts_event_parsed IS NULL
            THEN 1
            ELSE 0
        END) AS null_ts_event,
    SUM(CASE
        WHEN ts_recv_parsed IS NULL
            THEN 1
            ELSE 0
        END) AS null_ts_recv
FROM mbo_time
"""

spark.sql(query).show()

+----------+-------------+------------+
|total_rows|null_ts_event|null_ts_recv|
+----------+-------------+------------+
| 382479376|            0|           0|
+----------+-------------+------------+


## Decode flag bits

According to Databento MBO schema, the `flags` column is a bit field where a
single integer value can contain multiple indicators. Therefore, we will
decode the bits into separate columns before filtering the data.

- `is_last` (128): last record in an event
- `is_tob` (64): top-of-book message
- `is_snapshot` (32): snapshot record
- `is_mbp` (16): market-by-price message
- `is_bad_ts_recv` (8): `ts_recv` is unreliable
- `is_maybe_bad_book` (4): possible unrecoverable gap in the channel
- `is_publisher_specific` (2): publisher-specific meaning

Remove `is_snapshot` rows to focus on live market activity.

In [ ]:
# decode important Databento flag bits

query = """
SELECT
    *,
    CASE
        WHEN (flags & 128) != 0
            THEN 1
            ELSE 0
    END AS is_last,
    CASE
        WHEN (flags &  64) != 0
            THEN 1
            ELSE 0
    END AS is_tob,
    CASE
        WHEN (flags &  32) != 0
            THEN 1
            ELSE 0
    END AS is_snapshot,
    CASE
        WHEN (flags &  16) != 0
            THEN 1
            ELSE 0
    END AS is_mbp,
    CASE
        WHEN (flags &   8) != 0
            THEN 1
            ELSE 0
    END AS is_bad_ts_recv,
    CASE
        WHEN (flags &   4) != 0
            THEN 1
            ELSE 0
    END AS is_maybe_bad_book,
    CASE
        WHEN (flags &   2) != 0
            THEN 1
            ELSE 0
    END AS is_publisher_specific
FROM mbo_time
"""

flagged_sdf = spark.sql(query)
flagged_sdf.createOrReplaceTempView("mbo_flagged")

In [ ]:
# check flags
query = """
SELECT
    SUM(is_last) AS last_rows,
    SUM(is_snapshot) AS snapshot_rows,
    SUM(is_bad_ts_recv) AS bad_ts_recv_rows,
    SUM(is_maybe_bad_book) AS maybe_bad_book_rows,
    SUM(is_publisher_specific) AS publisher_specific_rows,
    SUM(is_mbp) AS mbp_rows,
    SUM(is_tob) AS tob_rows
FROM mbo_flagged
"""

spark.sql(query).show()

+---------+-------------+----------------+-------------------+-----------------------+--------+--------+
|last_rows|snapshot_rows|bad_ts_recv_rows|maybe_bad_book_rows|publisher_specific_rows|mbp_rows|tob_rows|
+---------+-------------+----------------+-------------------+-----------------------+--------+--------+
|324591268|      2505529|         2505614|            1150122|                      0|       0|       0|
+---------+-------------+----------------+-------------------+-----------------------+--------+--------+


## Filter snapshot and administrative rows

Filter out rows that are not live market activity.

This includes:
- snapshot rows identified by the Databento `F_SNAPSHOT` flag
- `R` and `N` rows that are not part of live market activity

In [ ]:
# keep only live market event rows
query = """
SELECT
    *
FROM mbo_flagged
WHERE
    is_snapshot = 0
AND
    action NOT IN ('R', 'N')
"""

filtered_sdf = spark.sql(query)
filtered_sdf.createOrReplaceTempView("mbo_filtered")

In [ ]:
# sanity check
query = """
SELECT
    action,
    COUNT(*) AS action_count
FROM mbo_filtered
GROUP BY
    action
ORDER BY
    action_count DESC
"""

spark.sql(query).show()

+------+------------+
|action|action_count|
+------+------------+
|     C|   140791991|
|     A|   140745029|
|     M|    63103284|
|     F|    26429373|
|     T|     8904064|
+------+------------+


In [ ]:
# row count after filtering
query = """
SELECT
    COUNT(*) AS filtered_rows
FROM mbo_filtered
"""

spark.sql(query).show()

+-------------+
|filtered_rows|
+-------------+
|    379973741|
+-------------+


After filtering snapshot and administrative records and checking for null values
the glbx-mdp3-20251201-20251215.mbo.csv dataset contains 43,329,745 rows.

After filtering snapshot and administrative records and checking for null values
the glbx-mdp3-20251216-20251231.mbo.csv dataset contains 47,188,552 rows.

After filtering snapshot and administrative records and checking for null values
the glbx-mdp3-20251115-20251130.mbo.csv dataset contains 134,420,783 rows.

After filtering snapshot and administrative records and checking for null values
the glbx-mdp3-20260101-20260115.mbo.csv dataset contains  155,034,661 rows.

Total cleaned rows: 379,973,741

## Aggregate into 1-second buckets

Assign each event for each symbol to a 1-second bucket, then
aggregate the events. Each row represents events for one symbol in one second.

The resulting table is a 1-second event summary with:
- `n_events` - total events
- `n_add` - number of add orders
- `n_cancel` - number of cancel orders
- `n_modify` - number of modify orders
- `n_trade` - number of trade records (`T`)
- `n_fill` - number of fill records (`F`)
- `n_add_bid` - number of add orders on the bid side
- `n_add_ask` - number of add orders on the ask side
- `n_cancel_bid` - number of cancel orders on the bid side
- `n_cancel_ask` - number of cancel orders on the ask side
- `n_modify_bid` - number of modify orders on the bid side
- `n_modify_ask` - number of modify orders on the ask side
- `n_trade_buy` - number of trade records on the buy side
- `n_trade_sell` - number of trade records on the sell side
- `n_trade_none` - number of trade records with no side specified
- `trade_volume_buy` - total trade size on the buy side
- `trade_volume_sell` - total trade size on the sell side
- `trade_volume_none` - total trade size with no side specified
- `trade_volume_total` - total trade size
- `fill_volume_total` - total fill size

In [ ]:
# truncate event timestamps to the second so events occurring in the same second share one bucket
query = """
SELECT
    *,
    DATE_TRUNC('second', ts_event_parsed) AS second_bucket
FROM mbo_filtered
"""

# persist views when appropriate
from pyspark.storagelevel import StorageLevel

bucketed_sdf = spark.sql(query).persist(StorageLevel.MEMORY_AND_DISK)
bucketed_sdf.createOrReplaceTempView("mbo_bucketed")

In [ ]:
# sanity check
query = """
SELECT
    ts_event,
    ts_event_parsed,
    second_bucket
FROM mbo_bucketed
LIMIT 10
"""

spark.sql(query).show(truncate=False)

+------------------------------+--------------------------+-------------------+
|ts_event                      |ts_event_parsed           |second_bucket      |
+------------------------------+--------------------------+-------------------+
|2025-11-16T22:00:00.186484811Z|2025-11-16 22:00:00.186484|2025-11-16 22:00:00|
|2025-11-16T22:00:00.186560687Z|2025-11-16 22:00:00.18656 |2025-11-16 22:00:00|
|2025-11-16T22:00:00.186592115Z|2025-11-16 22:00:00.186592|2025-11-16 22:00:00|
|2025-11-16T22:00:00.186619051Z|2025-11-16 22:00:00.186619|2025-11-16 22:00:00|
|2025-11-16T22:00:00.186620587Z|2025-11-16 22:00:00.18662 |2025-11-16 22:00:00|
|2025-11-16T22:00:00.186637731Z|2025-11-16 22:00:00.186637|2025-11-16 22:00:00|
|2025-11-16T22:00:00.186884671Z|2025-11-16 22:00:00.186884|2025-11-16 22:00:00|
|2025-11-16T22:00:00.186971317Z|2025-11-16 22:00:00.186971|2025-11-16 22:00:00|
|2025-11-16T22:00:00.187050669Z|2025-11-16 22:00:00.18705 |2025-11-16 22:00:00|
|2025-11-16T22:00:00.187300141Z|2025-11-

In [ ]:
# group by symbol and 1-second bucket, then aggregate events
# one row per contract per second that an event occurs

query = """
SELECT
    symbol,
    second_bucket,
    COUNT(*) AS n_events,
    SUM(CASE
        WHEN action = 'A'
            THEN 1
            ELSE 0
    END) AS n_add,
    SUM(CASE
        WHEN action = 'C'
            THEN 1
            ELSE 0
    END) AS n_cancel,
    SUM(CASE
        WHEN action = 'M'
            THEN 1
            ELSE 0
    END) AS n_modify,
    SUM(CASE
        WHEN action = 'T'
            THEN 1
            ELSE 0
    END) AS n_trade,
    SUM(CASE
        WHEN action = 'F'
            THEN 1
            ELSE 0
    END) AS n_fill,
    SUM(CASE
        WHEN action = 'A' AND side = 'B'
            THEN 1
            ELSE 0
    END) AS n_add_bid,
    SUM(CASE
        WHEN action = 'A' AND side = 'A'
            THEN 1
            ELSE 0
    END) AS n_add_ask,
    SUM(CASE
        WHEN action = 'C' AND side = 'B'
            THEN 1
            ELSE 0
    END) AS n_cancel_bid,
    SUM(CASE
        WHEN action = 'C' AND side = 'A'
            THEN 1
            ELSE 0
    END) AS n_cancel_ask,
    SUM(CASE
        WHEN action = 'M' AND side = 'B'
            THEN 1
            ELSE 0
    END) AS n_modify_bid,
    SUM(CASE
        WHEN action = 'M' AND side = 'A'
            THEN 1
            ELSE 0
    END) AS n_modify_ask,
    SUM(CASE
        WHEN action = 'T' AND side = 'B'
            THEN 1
            ELSE 0
    END) AS n_trade_buy,
    SUM(CASE
        WHEN action = 'T' AND side = 'A'
            THEN 1
            ELSE 0
    END) AS n_trade_sell,
    SUM(CASE
        WHEN action = 'T' AND side = 'N'
            THEN 1
            ELSE 0
    END) AS n_trade_none,
    SUM(CASE
        WHEN action = 'T' AND side = 'B'
            THEN size
            ELSE 0
    END) AS trade_volume_buy,
    SUM(CASE
        WHEN action = 'T' AND side = 'A'
            THEN size
            ELSE 0
    END) AS trade_volume_sell,
    SUM(CASE
        WHEN action = 'T' AND side = 'N'
            THEN size
            ELSE 0
    END) AS trade_volume_none,
    SUM(CASE
        WHEN action = 'T'
            THEN size
            ELSE 0
    END) AS trade_volume_total,
    SUM(CASE
        WHEN action = 'F'
            THEN size
            ELSE 0
    END) AS fill_volume_total
FROM mbo_bucketed
GROUP BY
    symbol,
    second_bucket
"""

sec_events_sdf = spark.sql(query).persist(StorageLevel.MEMORY_AND_DISK)
sec_events_sdf.createOrReplaceTempView("sec_events")

In [ ]:
# sanity check

query = """
SELECT
    symbol,
    second_bucket,
    n_events,
    n_add,
    n_cancel,
    n_modify,
    n_trade,
    n_fill
FROM sec_events
WHERE
    symbol = '6E.c.0'
ORDER BY
    second_bucket
LIMIT 20
"""

spark.sql(query).show(truncate=False)

+------+-------------------+--------+-----+--------+--------+-------+------+
|symbol|second_bucket      |n_events|n_add|n_cancel|n_modify|n_trade|n_fill|
+------+-------------------+--------+-----+--------+--------+-------+------+
|6E.c.0|2025-11-16 22:55:00|1       |0    |1       |0       |0      |0     |
|6E.c.0|2025-11-16 23:02:25|2       |2    |0       |0       |0      |0     |
|6E.c.0|2025-11-16 23:02:30|1       |1    |0       |0       |0      |0     |
|6E.c.0|2025-11-16 23:02:31|1       |0    |1       |0       |0      |0     |
|6E.c.0|2025-11-16 23:02:52|1       |1    |0       |0       |0      |0     |
|6E.c.0|2025-11-16 23:02:53|13      |6    |7       |0       |0      |0     |
|6E.c.0|2025-11-16 23:02:55|2       |1    |1       |0       |0      |0     |
|6E.c.0|2025-11-16 23:02:56|12      |6    |6       |0       |0      |0     |
|6E.c.0|2025-11-16 23:02:58|2       |1    |1       |0       |0      |0     |
|6E.c.0|2025-11-16 23:02:59|2       |1    |1       |0       |0      |0     |

In [ ]:
# count number of rows after grouping
query = """
SELECT
    COUNT(*) AS row_count
FROM sec_events
"""

spark.sql(query).show()

+---------+
|row_count|
+---------+
| 12838370|
+---------+


Number of rows after bucketing glbx-mdp3-20251201-20251215.mbo.csv:  2,337,510

Number of rows after bucketing glbx-mdp3-20251216-20251231.mbo.csv: 2,364,256

Number of rows after bucketing glbx-mdp3-20251115-20251130.mbo.csv: 3,909,795

Number of rows after bucketing glbx-mdp3-20260101-20260115.mbo.csv: 4,226,809

Total rows = 12,838,370

## Create and join side/size features

Engineer additional 1-second features from `side` and `size`
for each `symbol` and `second_bucket`.

The side/size features include:
- `n_bid` - number of bid-side events
- `n_ask` - number of ask-side events
- `bid_size` - total bid-side size
- `ask_size` - total ask-side size
- `avg_size` - average event size
- `sum_size` - total event size
- `max_size` - largest event size

The features are then joined with the sec_events table from the last step.

In [ ]:
# aggregate side and size features for each symbol-second bucket

query = """
SELECT
    symbol,
    second_bucket,
    SUM(CASE
            WHEN side = 'B'
                THEN 1
                ELSE 0
        END) AS n_bid,
    SUM(CASE
            WHEN side = 'A'
                THEN 1
                ELSE 0
        END) AS n_ask,
    SUM(CASE
            WHEN side = 'B'
                THEN size
                ELSE 0
        END) AS bid_size,
    SUM(CASE
            WHEN side = 'A'
                THEN size
                ELSE 0
        END) AS ask_size,
    AVG(size) AS avg_size,
    SUM(size) AS sum_size,
    MAX(size) AS max_size
FROM mbo_bucketed
GROUP BY
    symbol,
    second_bucket
"""

side_size_sdf = spark.sql(query)
side_size_sdf.createOrReplaceTempView("side_size")

In [ ]:
# sanity check
query = """
SELECT
    *
FROM side_size
LIMIT 10
"""

spark.sql(query).show()

+------+-------------------+-----+-----+--------+--------+------------------+--------+--------+
|symbol|      second_bucket|n_bid|n_ask|bid_size|ask_size|          avg_size|sum_size|max_size|
+------+-------------------+-----+-----+--------+--------+------------------+--------+--------+
|ZF.c.0|2025-11-17 13:02:23|  487|  279|    7466|    4939|       16.16015625|   12411|     741|
|CL.c.0|2025-11-17 13:08:07|    3|    1|       3|       1|               1.0|       4|       1|
|ZT.c.0|2025-11-17 13:09:49|    8|    4|      30|      25| 4.583333333333333|      55|      17|
|ZB.c.0|2025-11-17 13:10:06|   17|    8|      19|      12|              1.24|      31|       3|
|ZB.c.0|2025-11-17 13:10:52|   49|  135|     606|    1890|13.565217391304348|    2496|     260|
|ZB.c.0|2025-11-17 13:12:51|    0|    1|       0|       1|               1.0|       1|       1|
|TN.c.0|2025-11-17 13:20:15|   19|    2|     381|      16|18.904761904761905|     397|      78|
|ZF.c.0|2025-11-17 13:21:07|   11|    2|

In [ ]:
# join side and size features to sec_events table

query = """
SELECT
    e.symbol,
    e.second_bucket,
    e.n_events,
    e.n_add,
    e.n_cancel,
    e.n_modify,
    e.n_trade,
    e.n_fill,
    e.n_add_bid,
    e.n_add_ask,
    e.n_cancel_bid,
    e.n_cancel_ask,
    e.n_modify_bid,
    e.n_modify_ask,
    e.n_trade_buy,
    e.n_trade_sell,
    e.n_trade_none,
    e.trade_volume_buy,
    e.trade_volume_sell,
    e.trade_volume_none,
    e.trade_volume_total,
    e.fill_volume_total,
    s.n_bid,
    s.n_ask,
    s.bid_size,
    s.ask_size,
    s.avg_size,
    s.sum_size,
    s.max_size
FROM sec_events e
JOIN side_size s
    ON e.symbol = s.symbol
    AND e.second_bucket = s.second_bucket
"""



features_sdf = spark.sql(query).persist(StorageLevel.MEMORY_AND_DISK)
features_sdf.createOrReplaceTempView("features")


In [ ]:
# row count after joining side and size features
query = """
SELECT
    COUNT(*) AS n_rows
FROM features
"""

spark.sql(query).show()

+--------+
|  n_rows|
+--------+
|12838370|
+--------+


In [ ]:
# May need to unpersist old views depending on memory constraints of your compute cluster
#sec_events_sdf.unpersist()

## Create trade-only last price column and join with feature table

Create a 1-second price series using only trade rows,
i.e. `action = 'T'`, so it is based on executed transactions only.

For each `symbol` and `second_bucket`,
select the last trade price using a deterministic ordering.

Then use a `LEFT JOIN` to combine the feature table with the
trade-price table so all feature rows are kept.

In [ ]:
# get last trade price for each symbol per second from mbo_filtered

query = """
WITH trades AS (
    SELECT
        symbol,
        DATE_TRUNC('second', ts_event_parsed) AS second_bucket,
        price,
        ROW_NUMBER() OVER (
            PARTITION BY
                symbol,
                DATE_TRUNC('second', ts_event_parsed)
            ORDER BY
                ts_event DESC,
                channel_id DESC,
                sequence DESC,
                order_id DESC
        ) AS row_num
    FROM mbo_filtered
    WHERE
        action = 'T'
        AND price IS NOT NULL
)
SELECT
    symbol,
    second_bucket,
    price AS last_price
FROM trades
WHERE
    row_num = 1
"""

last_price_sdf = spark.sql(query)
last_price_sdf.createOrReplaceTempView("last_price")

In [ ]:
# join last_price into the feature table

query = """
SELECT
    f.symbol,
    f.second_bucket,
    f.n_events,
    f.n_add,
    f.n_cancel,
    f.n_modify,
    f.n_trade,
    f.n_fill,
    f.n_add_bid,
    f.n_add_ask,
    f.n_cancel_bid,
    f.n_cancel_ask,
    f.n_modify_bid,
    f.n_modify_ask,
    f.n_trade_buy,
    f.n_trade_sell,
    f.n_trade_none,
    f.trade_volume_buy,
    f.trade_volume_sell,
    f.trade_volume_none,
    f.trade_volume_total,
    f.fill_volume_total,
    f.n_bid,
    f.n_ask,
    f.bid_size,
    f.ask_size,
    f.avg_size,
    f.sum_size,
    f.max_size,
    p.last_price
FROM features f
LEFT JOIN last_price p
    ON f.symbol = p.symbol
    AND f.second_bucket = p.second_bucket
"""


features_last_price_sdf = spark.sql(query).persist(StorageLevel.MEMORY_AND_DISK)
features_last_price_sdf.createOrReplaceTempView("features_last_price")

In [ ]:
# confirm feature row count

query = """
SELECT
    COUNT(*) AS feature_rows
FROM features_last_price
"""

spark.sql(query).show()

+------------+
|feature_rows|
+------------+
|    12838370|
+------------+


In [ ]:
# May need to unpersist old views depending on memory constraints of your compute cluster
#features_sdf.unpersist()

## Build 1-second trading session timeline

The combined features table is sparse because many seconds have no events.  
This is a problem for 60-second clock-time calculations, since we want one row for every second.

To fix this, we assign each row to a CME trading session and build a full 1-second timeline
for each observed symbol-session using the standard-time (CST-based) UTC session schedule:
- starts at 23:00:00 UTC on the prior date
- ends at 21:59:59 UTC on the session date
- excludes the 22:00–22:59 UTC maintenance hour

Steps include:
1. assign `session_date` to each sparse row
2. identify all symbol-session pairs
3. generate one row for every second in each session
4. join the features table onto that timeline
5. fill event and size features with zeros, while leaving `last_price` as null for now
6. remove holiday-affected and incomplete sessions

This gives a full second-by-second table for each observed symbol-session.

In [ ]:
# assign CME session_date to each row in features table
# 23:00 hour belongs to the next session date
# 00:00-21:59 belongs to the same session date
# 22:00 hour is excluded as the maintenance window

query = """
SELECT
    *,
    CASE
        WHEN HOUR(second_bucket) = 23 THEN DATE_ADD(TO_DATE(second_bucket), 1)
        WHEN HOUR(second_bucket) BETWEEN 0 AND 21 THEN TO_DATE(second_bucket)
        ELSE NULL
    END AS session_date
FROM features_last_price
WHERE
    HOUR(second_bucket) != 22
"""

session_times_sdf = spark.sql(query)
session_times_sdf.createOrReplaceTempView("session_times")

In [ ]:
# keep only weekday session dates
# assign official session start and end times

query = """
WITH session_days AS (
    SELECT DISTINCT
        symbol,
        session_date
    FROM session_times
    WHERE
        session_date IS NOT NULL
        AND DAYOFWEEK(session_date) BETWEEN 2 AND 6
)
SELECT
    symbol,
    session_date,
    TO_TIMESTAMP(CONCAT(CAST(DATE_SUB(session_date, 1) AS STRING), ' 23:00:00')) AS session_start,
    TO_TIMESTAMP(CONCAT(CAST(session_date AS STRING), ' 21:59:59')) AS session_end
FROM session_days
"""

sessions_sdf = spark.sql(query)
sessions_sdf.createOrReplaceTempView("sessions")

In [ ]:
# create one row for every second in every observed session for each symbol

query = """
SELECT
    symbol,
    session_date,
    EXPLODE(
        SEQUENCE(
            session_start,
            session_end,
            INTERVAL 1 SECOND
        )
    ) AS second_bucket
FROM sessions
"""

session_secs_sdf = spark.sql(query).persist(StorageLevel.MEMORY_AND_DISK)
session_secs_sdf.createOrReplaceTempView("session_secs")

In [ ]:
# join features table onto the dense timeline
# fill activity and size columns with 0
# keep last_price as null for now

query = """
SELECT
    s.symbol,
    s.session_date,
    s.second_bucket,
    COALESCE(f.n_events, 0) AS n_events,
    COALESCE(f.n_add, 0) AS n_add,
    COALESCE(f.n_cancel, 0) AS n_cancel,
    COALESCE(f.n_modify, 0) AS n_modify,
    COALESCE(f.n_trade, 0) AS n_trade,
    COALESCE(f.n_fill, 0) AS n_fill,
    COALESCE(f.n_add_bid, 0) AS n_add_bid,
    COALESCE(f.n_add_ask, 0) AS n_add_ask,
    COALESCE(f.n_cancel_bid, 0) AS n_cancel_bid,
    COALESCE(f.n_cancel_ask, 0) AS n_cancel_ask,
    COALESCE(f.n_modify_bid, 0) AS n_modify_bid,
    COALESCE(f.n_modify_ask, 0) AS n_modify_ask,
    COALESCE(f.n_trade_buy, 0) AS n_trade_buy,
    COALESCE(f.n_trade_sell, 0) AS n_trade_sell,
    COALESCE(f.n_trade_none, 0) AS n_trade_none,
    COALESCE(f.trade_volume_buy, 0) AS trade_volume_buy,
    COALESCE(f.trade_volume_sell, 0) AS trade_volume_sell,
    COALESCE(f.trade_volume_none, 0) AS trade_volume_none,
    COALESCE(f.trade_volume_total, 0) AS trade_volume_total,
    COALESCE(f.fill_volume_total, 0) AS fill_volume_total,
    COALESCE(f.n_bid, 0) AS n_bid,
    COALESCE(f.n_ask, 0) AS n_ask,
    COALESCE(f.bid_size, 0) AS bid_size,
    COALESCE(f.ask_size, 0) AS ask_size,
    COALESCE(f.avg_size, 0) AS avg_size,
    COALESCE(f.sum_size, 0) AS sum_size,
    COALESCE(f.max_size, 0) AS max_size,
    f.last_price
FROM session_secs s
LEFT JOIN features_last_price f
    ON s.symbol = f.symbol
    AND s.second_bucket = f.second_bucket
"""


features_dense_sdf = spark.sql(query).persist(StorageLevel.MEMORY_AND_DISK)
features_dense_sdf.createOrReplaceTempView("features_dense")

In [ ]:
# row count

query = """
SELECT
    COUNT(*) AS rows
FROM features_dense
"""

spark.sql(query).show(truncate=False)

+--------+
|rows    |
+--------+
|32043600|
+--------+


In [ ]:
# May need to unpersist old views depending on memory constraints of your compute cluster
#features_last_price_sdf.unpersist()

## Forward-fill trade price and create the previous-price column

The `last_price` column only has values in rows where an executed trade occurred.  
For seconds with no executed trade, we forward-fill the most recent observed trade
price within each `symbol` and `session_date` to create `filled_price`.

We also create `prev_price` by taking the filled trade price from the previous second.


These columns are needed to compute the 1-second log return:
- `filled_price` = current second's trade price
- `prev_price` = previous second's trade price

In [ ]:
# forward-fill the last observed trade price within each symbol and session_date

query = """
SELECT
    *,
    LAST_VALUE(last_price, TRUE)
        OVER (
            PARTITION BY
                symbol,
                session_date
            ORDER BY
                second_bucket
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS filled_price
FROM features_dense
"""

features_fill_sdf = spark.sql(query)
features_fill_sdf.createOrReplaceTempView("features_fill")

In [ ]:
# get the previous second's filled trade price

query = """
SELECT
    *,
    LAG(filled_price) OVER (
        PARTITION BY
            symbol,
            session_date
        ORDER BY
            second_bucket
    ) AS prev_price
FROM features_fill
"""

features_lag_sdf = spark.sql(query)
features_lag_sdf.createOrReplaceTempView("features_lag")

## Compute the 1-second log return

Compute the 1-second log return using the filled current price and the previous second's price:

`log_return = ln(filled_price / prev_price)`

The first row for each symbol by session_date partition has no previous price, so its log return is `NULL`.

We use log returns rather than simple returns because they are additive over
time, symmetric around zero, and the standard basis for realized volatility.

In [ ]:
# compute the 1-second log return from the filled price series

query = """
SELECT
    *,
    CASE
        WHEN
            filled_price > 0
            AND prev_price > 0
                THEN LN(filled_price / prev_price)
                ELSE NULL
    END AS log_return
FROM features_lag
"""

features_log_sdf = spark.sql(query)
features_log_sdf.createOrReplaceTempView("features_log")

## Compute past 60-second, future 60-second, and future 300-second realized volatility

Use the 1-second log returns to compute realized volatility over rolling windows.

For each row, we create:

- `rv_past_60`: realized volatility over the previous 60 seconds, including the current second
- `rv_next_60`: realized volatility over the next 60 seconds  (label for future models)
- `rv_next_300`: realized volatility over the next 300 seconds, or 5 minutes (label for future models)

All are computed as the square root of the sum of squared 1-second log returns within the corresponding window.

Seconds with no price change have `log_return = 0`, so they contribute zero to realized volatility.

All calculations stay within each partition. Therefore, we focus on within-session short-horizon volatility by
preventing windows from crossing CME maintenance breaks or between-session gaps.


In [ ]:

# compute realized volatility over the past 60 seconds,
# next 60 seconds, and next 5 minutes

query = """
SELECT
    *,
    SQRT(SUM(POWER(COALESCE(log_return, 0.0), 2)) OVER (
        PARTITION BY
            symbol,
            session_date
        ORDER BY
            second_bucket
        ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
    )) AS rv_past_60,
    SQRT(SUM(POWER(COALESCE(log_return, 0.0), 2)) OVER (
        PARTITION BY
            symbol,
            session_date
        ORDER BY
            second_bucket
        ROWS BETWEEN 1 FOLLOWING AND 60 FOLLOWING
    )) AS rv_next_60,
    SQRT(SUM(POWER(COALESCE(log_return, 0.0), 2)) OVER (
        PARTITION BY
            symbol,
            session_date
        ORDER BY
            second_bucket
        ROWS BETWEEN 1 FOLLOWING AND 300 FOLLOWING
    )) AS rv_next_300
FROM features_log
"""

features_rv_sdf = spark.sql(query).persist(StorageLevel.MEMORY_AND_DISK)
features_rv_sdf.createOrReplaceTempView("features_rv")

### Create rolling 60-second features



Compute rolling 60-second features within each symbol by session_date partition, including:
- rolling order book activity (n_events_60, n_add_60, n_cancel_60, n_modify_60, n_trade_60, n_fill_60)
- rolling size / volume (sum_size_60)
- rolling imbalance (order_imbalance_60, size_imbalance_60)
- rolling return dispersion (ret_std_60), i.e. standard deviation of 1-second log returns over the past 60 seconds

Drop:
- rows without a full 60-second lookback
- rows without a valid future 60-second target
- rows without a valid future 300-second target

We will use 60-second rolling features to help predict future 60-second and 5-minute realized volatility.

In [ ]:
# create rolling 60-second features
# keep only rows with a full 60-second lookback,
# full 60-second future horizon, and full 5-minute future horizon

query = """
WITH rolling AS (
    SELECT
        *,
        SUM(n_events) OVER past_window AS n_events_60,
        SUM(n_add) OVER past_window AS n_add_60,
        SUM(n_cancel) OVER past_window AS n_cancel_60,
        SUM(n_modify) OVER past_window AS n_modify_60,
        SUM(n_trade) OVER past_window AS n_trade_60,
        SUM(n_fill) OVER past_window AS n_fill_60,
        SUM(sum_size) OVER past_window AS sum_size_60,
        SUM(n_bid - n_ask) OVER past_window AS order_imbalance_60,
        SUM(bid_size - ask_size) OVER past_window AS size_imbalance_60,
        STDDEV_POP(COALESCE(log_return, 0.0)) OVER past_window AS ret_std_60,
        COUNT(*) OVER past_window AS n_past_60,
        COUNT(*) OVER future_window_60 AS n_next_60,
        COUNT(*) OVER future_window_300 AS n_next_300
    FROM features_rv
    WINDOW
        past_window AS (
            PARTITION BY
                symbol,
                session_date
            ORDER BY
                second_bucket
            ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
        ),
        future_window_60 AS (
            PARTITION BY
                symbol,
                session_date
            ORDER BY
                second_bucket
            ROWS BETWEEN 1 FOLLOWING AND 60 FOLLOWING
        ),
        future_window_300 AS (
            PARTITION BY
                symbol,
                session_date
            ORDER BY
                second_bucket
            ROWS BETWEEN 1 FOLLOWING AND 300 FOLLOWING
        )
)
SELECT
    *
FROM rolling
WHERE
    n_past_60 = 60
    AND n_next_60 = 60
    AND n_next_300 = 300
    AND filled_price IS NOT NULL
    AND prev_price IS NOT NULL
    AND log_return IS NOT NULL
"""

features_rolling_sdf = spark.sql(query).persist(StorageLevel.MEMORY_AND_DISK)
features_rolling_sdf.createOrReplaceTempView("features_rolling")

In [ ]:
# final inspection before saving
query = """
SELECT
    *
FROM features_rolling
WHERE
    symbol = '6E.c.0'
    AND filled_price IS NOT NULL
    AND prev_price IS NOT NULL
    AND log_return IS NOT NULL
    AND rv_past_60 > 0
    AND rv_next_60 > 0
    AND rv_next_300 > 0
    AND n_trade_60 > 0
ORDER BY
  session_date,
  second_bucket
LIMIT 100
"""

spark.sql(query).show(truncate=False)



+------+------------+-------------------+--------+-----+--------+--------+-------+------+---------+---------+------------+------------+------------+------------+-----------+------------+------------+----------------+-----------------+-----------------+------------------+-----------------+-----+-----+--------+--------+--------+--------+--------+----------+------------+----------+--------------------+--------------------+--------------------+--------------------+-----------+--------+-----------+-----------+----------+---------+-----------+------------------+-----------------+--------------------+---------+---------+----------+
|symbol|session_date|second_bucket      |n_events|n_add|n_cancel|n_modify|n_trade|n_fill|n_add_bid|n_add_ask|n_cancel_bid|n_cancel_ask|n_modify_bid|n_modify_ask|n_trade_buy|n_trade_sell|n_trade_none|trade_volume_buy|trade_volume_sell|trade_volume_none|trade_volume_total|fill_volume_total|n_bid|n_ask|bid_size|ask_size|avg_size|sum_size|max_size|last_price|filled_pri

In [ ]:
# move both target columns to the end
target_cols = ["rv_next_60", "rv_next_300"]

other_cols = [c for c in features_rolling_sdf.columns if c not in target_cols]
features_final_sdf = features_rolling_sdf.select(*other_cols, *target_cols)


In [ ]:
# drop helper columns
features_final_sdf = features_final_sdf.drop("n_past_60", "n_next_60", "n_next_300")


In [ ]:
print(features_final_sdf.columns)

['symbol', 'session_date', 'second_bucket', 'n_events', 'n_add', 'n_cancel', 'n_modify', 'n_trade', 'n_fill', 'n_add_bid', 'n_add_ask', 'n_cancel_bid', 'n_cancel_ask', 'n_modify_bid', 'n_modify_ask', 'n_trade_buy', 'n_trade_sell', 'n_trade_none', 'trade_volume_buy', 'trade_volume_sell', 'trade_volume_none', 'trade_volume_total', 'fill_volume_total', 'n_bid', 'n_ask', 'bid_size', 'ask_size', 'avg_size', 'sum_size', 'max_size', 'last_price', 'filled_price', 'prev_price', 'log_return', 'rv_past_60', 'n_events_60', 'n_add_60', 'n_cancel_60', 'n_modify_60', 'n_trade_60', 'n_fill_60', 'sum_size_60', 'order_imbalance_60', 'size_imbalance_60', 'ret_std_60', 'rv_next_60', 'rv_next_300']


In [ ]:
#persist and materialize final table while getting final row count before write to S3

features_final_sdf = features_final_sdf.persist(StorageLevel.MEMORY_AND_DISK)
features_final_sdf.count()

np.int64(28681716)

## Save final features table to S3

Save the features table to S3 as a Parquet dataset partitioned by `symbol`.

Dataset can then be loaded either:
- all at once for cross-asset analysis, or
- by individual symbol for symbol-level modeling and evaluation

In [ ]:
# output folder
S3_OUT = f"s3://{BUCKET}/processed_data/mbo_features_table"

# save parquet dataset partitioned by symbols
features_final_sdf.write.mode("overwrite").partitionBy("symbol").parquet(S3_OUT)

print("Saved dataset to:")
print(S3_OUT)

Saved dataset to:
s3://cis545-mbo-data-596916982631-us-west-2-an/processed_data/mbo_features_table


#final row count: 28,681,716